# Dynamic Programming — Subtopic 4

## DP on Subsequences (Knapsack family)

**Problems covered:**
1. Subset Sum (existence)
2. Partition Equal Subset Sum
3. Count of Subsets with Given Sum
4. Minimum Subset Sum Difference
5. Count of Subsets with Given Difference
6. 0/1 Knapsack
7. Unbounded Knapsack
8. Rod Cutting
9. Coin Change (min coins)
10. Coin Change II (count ways)

> Every problem here reduces to one of two skeletons — **0/1** or **unbounded** — and a choice of combine operator. The central conceptual axis is the **0/1 vs unbounded invariant shift**: a one-line change in the recurrence that flips the direction of space optimization.

---

# The state — `(item-index, capacity)` and why both are necessary

All ten problems share the same state shape:

$$\text{dp}[i][w] = \text{(some aggregate)}\ \text{over subsets of items in}\ \{0, 1, \ldots, i\}\ \text{achieving target}\ w$$

Concretely, depending on the problem flavor:

| Flavor | $\text{dp}[i][w]$ stands for |
|---|---|
| Existence (Subset Sum) | true ⟺ some subset of items $[0..i]$ sums to exactly $w$ |
| Counting (Count Subsets, Coin Change II) | number of subsets (or combinations) of items $[0..i]$ summing to $w$ |
| Optimization (Knapsack, Coin Change) | best value / min count achievable with items $[0..i]$ within capacity / target $w$ |

**Why both dimensions are mandatory.** Suppose we tried 1D with just $\text{dp}[w]$. Then the recurrence "take item $i$ or not" would need to know *whether item $i$ has already been used*. With $\text{dp}[w]$ alone, we cannot tell — the recursion would happily re-take the same item. The $i$ dimension is what enforces "subset" (each item at most once) by partitioning the search into "items up to $i$" vs "item $i+1$ and beyond."

For unbounded variants, item repetition is *allowed*. We will see that this manifests not as a different state shape, but as a one-character change to the recurrence (using $\text{dp}[i][\ldots]$ vs $\text{dp}[i-1][\ldots]$ on the "take" branch).

---

## The 0/1 vs Unbounded invariant — the central delta

Every problem in this subtopic resolves into one of these two skeletons.

**0/1 skeleton (each item used at most once):**
$$\boxed{\text{dp}[i][w] = \text{combine}\!\Big(\ \text{dp}[i-1][w]\ ,\ \ \text{transform}(\text{dp}[\mathbf{i-1}][w - \text{wt}[i]],\ \text{val}[i])\ \Big)}$$

**Unbounded skeleton (each item may be reused):**
$$\boxed{\text{dp}[i][w] = \text{combine}\!\Big(\ \text{dp}[i-1][w]\ ,\ \ \text{transform}(\text{dp}[\mathbf{i}][w - \text{wt}[i]],\ \text{val}[i])\ \Big)}$$

The **only** difference is the row index in the "take" branch: $i-1$ in 0/1 (item is consumed; we move to the previous item set) vs $i$ in unbounded (item remains available; we stay in the same item set with reduced capacity).

This one-character delta has a major consequence for space optimization.

---

## Space optimization direction — proof of why 0/1 iterates $w$ backwards, unbounded forwards

Both skeletons depend only on row $i-1$ (and possibly row $i$ at smaller $w$). So we can collapse to a single length-$(W{+}1)$ array $\text{dp}[w]$ and update it row by row.

**The danger:** when we update $\text{dp}[w]$, what we read at $\text{dp}[w - \text{wt}[i]]$ may be the **old** (row $i-1$) value or the **new** (row $i$) value depending on the iteration order on $w$.

**0/1 — must read old value → iterate $w$ from $W$ down to $\text{wt}[i]$.**
The recurrence reads $\text{dp}[i-1][w - \text{wt}[i]]$. After collapsing to 1D, $\text{dp}[w - \text{wt}[i]]$ holds the previous row's value **only if we have not yet processed it in this row**. If we iterate $w$ ascending, by the time we reach $w$ we have already overwritten $\text{dp}[w - \text{wt}[i]]$ — that's the new row value. Iterating $w$ descending preserves the previous-row value at smaller indices.

**Unbounded — must read new value → iterate $w$ from $\text{wt}[i]$ up to $W$.**
The recurrence reads $\text{dp}[i][w - \text{wt}[i]]$. This *is* the new row value — exactly what ascending iteration gives us (by the time we reach $w$, smaller indices have been updated to row $i$).

> **The mnemonic:** 0/1 reverses on $w$; unbounded goes forward on $w$. The reversal "protects" old values; the forward "exposes" new ones.

This is the most operationally important fact in subset-DP. Internalize it once and you will never need to re-derive it.

---

# Family A — Subset Sum variants (0/1, existence and counting)

This family is the cleanest entry into 2D DP on subsequences. All five problems use the same state $(i, s)$ where $s$ is a target sum. They differ only in the combine operator (OR for existence, $+$ for counting) and post-processing (final $\min$ for difference variants).

## 4.1 Subset Sum (existence)

**Problem.** Given non-negative integers $\text{nums}[0..n-1]$ and a target $T$, is there a subset of $\text{nums}$ summing to exactly $T$?

---

### Theory

**State definition.**
$\text{dp}[i][s]$ = true iff some subset of $\text{nums}[0..i]$ (using each element at most once) sums to exactly $s$.

**Invariant.** At every $(i, s)$, $\text{dp}[i][s]$ correctly answers the existence question over the first $i+1$ items. The recursion case-splits on the **last item considered, $\text{nums}[i]$**: either *exclude* it (then we need a subset of $\text{nums}[0..i-1]$ summing to $s$) or *include* it (then we need a subset summing to $s - \text{nums}[i]$, requiring $s \ge \text{nums}[i]$).

**Recurrence (0/1, existence flavor).**
$$\text{dp}[i][s] = \text{dp}[i-1][s]\ \vee\ \big(s \ge \text{nums}[i]\ \wedge\ \text{dp}[i-1][s - \text{nums}[i]]\big)$$

**Base cases.**
- $\text{dp}[i][0] = \text{true}$ for all $i \ge 0$ — the empty subset sums to $0$.
- $\text{dp}[0][s] = (s = \text{nums}[0])$ for $s \ge 1$ — with one item, only the single value is reachable (plus 0, handled above).

**Boundary transitions table.**

| Case | Recurrence value |
|---|---|
| $s = 0$ | true (empty subset) |
| $i = 0, s = \text{nums}[0]$ | true |
| $i = 0, s \ne 0$ and $s \ne \text{nums}[0]$ | false |
| $i \ge 1, s < \text{nums}[i]$ | $\text{dp}[i-1][s]$ (cannot include item $i$) |
| $i \ge 1, s \ge \text{nums}[i]$ | $\text{dp}[i-1][s] \vee \text{dp}[i-1][s - \text{nums}[i]]$ |

**Why it works.**
- *Optimal substructure (existence):* a subset of $\text{nums}[0..i]$ summing to $s$ either contains $\text{nums}[i]$ (and its complement is a subset of $\text{nums}[0..i-1]$ summing to $s - \text{nums}[i]$) or doesn't (and is itself a subset of $\text{nums}[0..i-1]$ summing to $s$). These two cases are exhaustive.
- *Overlapping subproblems:* naive recursion is $\Theta(2^n)$; the state space is $\Theta(n \cdot T)$.

**Complexity.** Time $O(n \cdot T)$, Space $O(n \cdot T)$ → $O(T)$.

**Space optimization — direction proof.**
The take case reads $\text{dp}[i-1][s - \text{nums}[i]]$ (previous row, smaller $s$). In the 1D collapse, this must be the **old** value. **Iterate $s$ from $T$ down to $\text{nums}[i]$** so smaller indices have not yet been overwritten in the current row. This is the 0/1 pattern.

**Delta.** This is the prototype 0/1 DP with **OR** combine. Every other problem in Family A is a variation of either the combine operator or the post-processing step.

---

In [ ]:
// Subset Sum — three implementations
#include <vector>
#include <iostream>
using namespace std;

// (A) Top-down memoization
//     dp[i][s] = true iff some subset of nums[0..i] sums to s.
//     Use int (0=false, 1=true, -1=unknown) instead of bool for sentinel.
int subsetSum_memo_helper(int i, int s, const vector<int>& nums,
                           vector<vector<int>>& dp) {
    if (s == 0) return 1;                                  // base: empty subset achieves sum 0
    if (i == 0) return (nums[0] == s) ? 1 : 0;             // single item: take it iff it matches s
    if (dp[i][s] != -1) return dp[i][s];
    int notTake = subsetSum_memo_helper(i - 1, s, nums, dp);
    int take = 0;
    if (s >= nums[i])                                      // guard: cannot include if it exceeds s
        take = subsetSum_memo_helper(i - 1, s - nums[i], nums, dp);
    dp[i][s] = (notTake || take) ? 1 : 0;                  // recurrence: OR combine
    return dp[i][s];
}
bool subsetSum_memo(const vector<int>& nums, int T) {
    int n = (int)nums.size();
    if (T == 0) return true;                               // empty subset always valid
    if (n == 0) return false;                              // empty array, T > 0: impossible
    vector<vector<int>> dp(n, vector<int>(T + 1, -1));
    return subsetSum_memo_helper(n - 1, T, nums, dp) == 1;
}

// (B) Tabulation — O(n*T)
bool subsetSum_tab(const vector<int>& nums, int T) {
    int n = (int)nums.size();
    if (T == 0) return true;
    if (n == 0) return false;
    vector<vector<bool>> dp(n, vector<bool>(T + 1, false));
    for (int i = 0; i < n; ++i) dp[i][0] = true;           // base: sum=0 always achievable
    if (nums[0] <= T) dp[0][nums[0]] = true;               // base: single item
    for (int i = 1; i < n; ++i) {
        for (int s = 1; s <= T; ++s) {
            bool notTake = dp[i-1][s];
            bool take = (s >= nums[i]) ? dp[i-1][s - nums[i]] : false;
            dp[i][s] = notTake || take;                    // recurrence
        }
    }
    return dp[n-1][T];
}

// (C) Space-optimized — O(T)
//     0/1 pattern: iterate s DESCENDING from T to nums[i].
//     This preserves dp[s - nums[i]] as the previous-row value when we read it.
bool subsetSum_opt(const vector<int>& nums, int T) {
    int n = (int)nums.size();
    if (T == 0) return true;
    if (n == 0) return false;
    vector<bool> dp(T + 1, false);
    dp[0] = true;                                          // base: sum=0
    if (nums[0] <= T) dp[nums[0]] = true;                  // base: single item
    for (int i = 1; i < n; ++i) {
        // DESCENDING iteration — see "Space optimization direction" in the opening framing.
        for (int s = T; s >= nums[i]; --s) {
            // dp[s] currently holds OLD value = dp[i-1][s] (not yet overwritten in this row)
            // dp[s - nums[i]] also holds OLD value (descending order protects it)
            dp[s] = dp[s] || dp[s - nums[i]];              // recurrence: OR
        }
        // For s < nums[i], we cannot take item i, so dp[s] = dp[i-1][s] = dp[s] (unchanged). Skipped.
    }
    return dp[T];
}


In [ ]:
// Tests — Subset Sum
auto run_ss = [](vector<int> nums, int T, bool expected) {
    bool a = subsetSum_memo(nums, T);
    bool b = subsetSum_tab(nums, T);
    bool c = subsetSum_opt(nums, T);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "T=" << T << " nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_ss({},               0, true);    // empty array, T=0: empty subset works
run_ss({},               5, false);   // empty array, T>0: impossible
run_ss({5},              0, true);    // T=0 always possible
run_ss({5},              5, true);    // single matching item
run_ss({5},              3, false);   // single non-matching item
run_ss({2, 3, 7, 8, 10}, 11, true);   // {3, 8} = 11
run_ss({2, 3, 7, 8, 10}, 14, false);  // unreachable
run_ss({1, 2, 3, 4, 5}, 15, true);    // sum of all
run_ss({1, 2, 3, 4, 5}, 16, false);   // exceeds total
run_ss({0, 0, 0},        0, true);    // all zeros, T=0
run_ss({1, 1, 1, 1},     3, true);    // 1+1+1


## 4.2 Partition Equal Subset Sum

**Problem.** Given non-negative integers $\text{nums}$, can we split them into two **disjoint** subsets $S_1, S_2$ with $\text{sum}(S_1) = \text{sum}(S_2)$ (and $S_1 \cup S_2 = \text{nums}$)?

---

### Theory — reduction to Subset Sum

Let $T = \sum_i \text{nums}[i]$.

> If $S_1 \cup S_2 = \text{nums}$ (disjoint) with $\text{sum}(S_1) = \text{sum}(S_2)$, then $\text{sum}(S_1) = T/2$.

**Two necessary conditions:**
1. $T$ must be even (else no half exists).
2. Some subset must sum to exactly $T/2$.

The answer to Partition Equal Subset Sum is **true** iff both hold. So this problem reduces *directly* to Subset Sum on target $T/2$.

**State, recurrence, base cases, complexity, space optimization** — *identical* to Subset Sum with $T \leftarrow T/2$.

**Delta.** No new DP machinery — it's a reduction. The pedagogical point is the **reduction technique**: many "partition" or "balance" problems first compute a candidate target from the total, then defer to subset sum.

---

In [ ]:
// Partition Equal Subset Sum — reduction to Subset Sum
#include <vector>
#include <numeric>
#include <iostream>
using namespace std;

bool canPartition(const vector<int>& nums) {
    int T = accumulate(nums.begin(), nums.end(), 0);
    if (T % 2 != 0) return false;                          // odd total: cannot split evenly
    return subsetSum_opt(nums, T / 2);                     // reuse the O(T)-space subset sum
}


In [ ]:
// Tests — Partition Equal Subset Sum
auto run_pess = [](vector<int> nums, bool expected) {
    bool a = canPartition(nums);
    cout << "nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> " << a << " (Expected: " << expected << ")"
         << (a == expected ? " OK" : " FAIL") << '\n';
};

run_pess({},                  true);   // empty: both halves are empty, both sum to 0
run_pess({1},                 false);  // odd total
run_pess({1, 1},              true);   // {1} and {1}
run_pess({1, 2, 3, 4},        true);   // {1,4} and {2,3}
run_pess({1, 5, 11, 5},       true);   // {1,5,5} and {11}, classic LeetCode case
run_pess({1, 2, 3, 5},        false);  // sum=11 odd
run_pess({1, 2, 5},           false);  // sum=8 even, but no subset sums to 4
run_pess({2, 2, 2, 2},        true);   // {2,2} and {2,2}
run_pess({100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100}, true);


## 4.3 Count of Subsets with Given Sum

**Problem.** Given non-negative integers $\text{nums}$ and a target $T$, count the number of subsets summing to exactly $T$.

---

### Theory

**State definition.**
$\text{dp}[i][s]$ = number of subsets of $\text{nums}[0..i]$ summing to exactly $s$.

**Invariant.** Same partition as Subset Sum — each subset is counted in exactly one of "contains $\text{nums}[i]$" or "doesn't contain $\text{nums}[i]$." So the count is additive across these two disjoint cases.

**Recurrence (0/1, counting flavor).**
$$\text{dp}[i][s] = \text{dp}[i-1][s] + \big(s \ge \text{nums}[i]\ ?\ \text{dp}[i-1][s - \text{nums}[i]]\ :\ 0\big)$$

**Base cases — subtle with zeros!**
- $\text{dp}[i][0]$ is generally **not** $1$.
- For $i = 0$:
  - If $\text{nums}[0] = 0$ and $s = 0$: there are **two** subsets summing to $0$ — $\{\}$ and $\{\text{nums}[0]\}$. So $\text{dp}[0][0] = 2$.
  - If $\text{nums}[0] > 0$: $\text{dp}[0][0] = 1$ (just the empty subset).
  - If $\text{nums}[0] \le T$ and $\text{nums}[0] > 0$: $\text{dp}[0][\text{nums}[0]] = 1$.
  - If $\text{nums}[0] = 0$: the above also requires care — but the unified base $\text{dp}[0][0] = 2$ handles it.

**Why the zero subtlety matters.** With element $0$, both "include" and "exclude" preserve the sum, so each $0$ in the array doubles the count of subsets achieving any given sum. Standard "set $\text{dp}[i][0] = 1$" loses this factor and gives the **wrong** answer.

**Boundary transitions table.**

| Case | Recurrence value |
|---|---|
| $i = 0, s = 0$ | $1$ if $\text{nums}[0] > 0$ else $2$ |
| $i = 0, s = \text{nums}[0] > 0$ | $1$ |
| $i = 0$, $s$ otherwise | $0$ |
| $i \ge 1, s < \text{nums}[i]$ | $\text{dp}[i-1][s]$ |
| $i \ge 1, s \ge \text{nums}[i]$ | $\text{dp}[i-1][s] + \text{dp}[i-1][s - \text{nums}[i]]$ |

**Complexity.** Time $O(n \cdot T)$, Space $O(n \cdot T)$ → $O(T)$.

**Space optimization.** Same as Subset Sum — iterate $s$ descending. (0/1 pattern.)

**Delta vs Subset Sum.** Same skeleton, OR → $+$. The trichotomy axis (existence → counting) changes the combine without changing the state. **Watch the zero base case.**

---

In [ ]:
// Count Subsets with Given Sum — three implementations
#include <vector>
#include <iostream>
using namespace std;

// (A) Top-down memoization
//     dp[i][s] = number of subsets of nums[0..i] summing to s.
int countSubsets_memo_helper(int i, int s, const vector<int>& nums,
                              vector<vector<int>>& dp) {
    if (i == 0) {
        // Handle the zero-element carefully:
        //   nums[0] = 0, s = 0  →  both include and exclude give sum 0 → 2 ways
        //   nums[0] > 0, s = 0  →  only exclude → 1 way
        //   s = nums[0] > 0     →  only include → 1 way
        //   else                →  0 ways
        if (s == 0 && nums[0] == 0) return 2;
        if (s == 0 || s == nums[0]) return 1;
        return 0;
    }
    if (dp[i][s] != -1) return dp[i][s];
    int notTake = countSubsets_memo_helper(i - 1, s, nums, dp);
    int take = 0;
    if (s >= nums[i])
        take = countSubsets_memo_helper(i - 1, s - nums[i], nums, dp);
    dp[i][s] = notTake + take;                             // recurrence: + combine
    return dp[i][s];
}
int countSubsets_memo(const vector<int>& nums, int T) {
    int n = (int)nums.size();
    if (n == 0) return (T == 0) ? 1 : 0;
    vector<vector<int>> dp(n, vector<int>(T + 1, -1));
    return countSubsets_memo_helper(n - 1, T, nums, dp);
}

// (B) Tabulation — O(n*T)
int countSubsets_tab(const vector<int>& nums, int T) {
    int n = (int)nums.size();
    if (n == 0) return (T == 0) ? 1 : 0;
    vector<vector<int>> dp(n, vector<int>(T + 1, 0));
    // base row i=0 — zero-aware
    if (nums[0] == 0) dp[0][0] = 2;                        // {} and {0}
    else { dp[0][0] = 1; if (nums[0] <= T) dp[0][nums[0]] = 1; }
    for (int i = 1; i < n; ++i) {
        for (int s = 0; s <= T; ++s) {
            int notTake = dp[i-1][s];
            int take = (s >= nums[i]) ? dp[i-1][s - nums[i]] : 0;
            dp[i][s] = notTake + take;                     // recurrence
        }
    }
    return dp[n-1][T];
}

// (C) Space-optimized — O(T)
//     0/1 pattern: iterate s DESCENDING.
int countSubsets_opt(const vector<int>& nums, int T) {
    int n = (int)nums.size();
    if (n == 0) return (T == 0) ? 1 : 0;
    vector<int> dp(T + 1, 0);
    if (nums[0] == 0) dp[0] = 2;
    else { dp[0] = 1; if (nums[0] <= T) dp[nums[0]] = 1; }
    for (int i = 1; i < n; ++i) {
        // DESCENDING preserves dp[s - nums[i]] as old (row i-1) value
        // But: with nums[i] = 0, the descending loop never enters (s >= nums[i] is s >= 0, full range).
        // For nums[i] = 0: dp[s] += dp[s - 0] = dp[s], so dp[s] = 2 * dp[s].
        //   Descending iteration on s is still correct (s - 0 = s, the diagonal — dp[s] read before write in same step is fine).
        for (int s = T; s >= nums[i]; --s) {
            dp[s] = dp[s] + dp[s - nums[i]];               // recurrence: notTake + take
        }
    }
    return dp[T];
}


In [ ]:
// Tests — Count Subsets with Given Sum
auto run_cs = [](vector<int> nums, int T, int expected) {
    int a = countSubsets_memo(nums, T);
    int b = countSubsets_tab(nums, T);
    int c = countSubsets_opt(nums, T);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "T=" << T << " nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_cs({},               0, 1);     // empty array, T=0: empty subset
run_cs({},               5, 0);     // empty array, T>0
run_cs({5},              5, 1);     // {5}
run_cs({5},              0, 1);     // {} only
run_cs({1, 1, 1},        2, 3);     // C(3,2) = 3 ways
run_cs({2, 3, 5, 6, 8, 10}, 10, 3); // {2,8} {2,3,5} {10}
run_cs({0, 0, 0, 0},     0, 16);    // each zero contributes a factor of 2: 2^4
run_cs({0, 1, 1},        1, 4);     // {1}, {0,1}, {1}, {0,1} — depends on indexing; let me recount: subsets summing to 1 are: {1@idx1}, {1@idx2}, {0,1@idx1}, {0,1@idx2} = 4
run_cs({1, 2, 3, 4, 5},  10, 3);    // {1,2,3,4}, {1,4,5}, {2,3,5}


## 4.4 Minimum Subset Sum Difference

**Problem.** Partition $\text{nums}$ into two subsets $S_1, S_2$ ($S_1 \cup S_2 = \text{nums}$, disjoint) minimizing $|\text{sum}(S_1) - \text{sum}(S_2)|$.

---

### Theory — reduction via Subset Sum's full row

Let $T = \sum \text{nums}$. For any partition, $\text{sum}(S_1) + \text{sum}(S_2) = T$, so

$$|\text{sum}(S_1) - \text{sum}(S_2)| = |T - 2 \cdot \text{sum}(S_1)|$$

We want to minimize this over reachable values of $\text{sum}(S_1)$. By symmetry of $S_1 \leftrightarrow S_2$, we can restrict to $\text{sum}(S_1) \in [0, T/2]$. So the problem becomes:

> Find the largest $s^* \in [0, T/2]$ such that some subset of $\text{nums}$ sums to $s^*$. The answer is $T - 2 s^*$.

**Algorithm.**
1. Run Subset Sum (existence) on all targets up to $T$.
2. Sweep $s$ from $0$ to $T/2$ and pick the largest reachable $s^*$.
3. Return $T - 2 s^*$.

**Crucial implementation note.** We need the **full row** $\text{dp}[n-1][\cdot]$ after the Subset Sum DP — not just $\text{dp}[n-1][T]$. The $O(T)$ space-optimized version already produces this row at the end (the array itself), so no extra work. We just don't return early.

**State, recurrence, complexity** — inherited from Subset Sum. **The novelty is the post-processing step.**

**Why iterate only up to $T/2$.** For any $s^* > T/2$, the complementary $T - s^*$ is in $[0, T/2)$ and gives the same $|T - 2 s^*|$. So we cover all distinct difference values within $[0, T/2]$. Iterating to $T/2$ saves half the post-processing work.

**Edge case.** If $\text{nums}$ contains negative values, the reduction breaks (sums can exceed $T$). All standard formulations assume non-negative inputs.

**Complexity.** Time $O(n \cdot T)$, Space $O(T)$.

**Delta.** Pure reduction — no new state. The lesson: when a subset-DP question concerns sums or differences over the whole array, often you compute the *reachability vector* once and post-process.

---

In [ ]:
// Minimum Subset Sum Difference
#include <vector>
#include <numeric>
#include <climits>
#include <iostream>
using namespace std;

int minSubsetSumDiff(const vector<int>& nums) {
    int n = (int)nums.size();
    if (n == 0) return 0;                                  // empty: |0-0| = 0
    int T = accumulate(nums.begin(), nums.end(), 0);

    // Build the reachable-sums vector using the 0/1 subset-sum DP (space-optimized).
    vector<bool> reach(T + 1, false);
    reach[0] = true;                                       // empty subset
    if (nums[0] <= T) reach[nums[0]] = true;
    for (int i = 1; i < n; ++i) {
        // 0/1 pattern: iterate s DESCENDING to protect the dp[s - nums[i]] read.
        for (int s = T; s >= nums[i]; --s) {
            if (reach[s - nums[i]]) reach[s] = true;       // OR combine
        }
    }

    // Post-process: largest s* in [0, T/2] with reach[s*] = true.
    // Answer is T - 2*s* (gap between halves).
    int best = INT_MAX;
    for (int s = 0; s <= T / 2; ++s) {
        if (reach[s]) {
            int diff = T - 2 * s;                          // since s <= T/2, diff >= 0
            if (diff < best) best = diff;
        }
    }
    return best;                                            // guaranteed valid since reach[0]=true
}


In [ ]:
// Tests — Minimum Subset Sum Difference
auto run_msd = [](vector<int> nums, int expected) {
    int a = minSubsetSumDiff(nums);
    cout << "nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> " << a << " (Expected: " << expected << ")"
         << (a == expected ? " OK" : " FAIL") << '\n';
};

run_msd({},               0);    // empty
run_msd({5},              5);    // single: one side is empty, diff = 5
run_msd({1, 1},           0);    // {1} | {1}
run_msd({1, 2, 3, 4},     0);    // {1,4} | {2,3}
run_msd({1, 6, 11, 5},    1);    // {1,5,6}=12 | {11}=11
run_msd({3, 1, 4, 2, 2},  0);    // total 12: split into 6 | 6
run_msd({1, 2, 7},        4);    // total 10: best split {7} | {1,2}=3 → diff 4
run_msd({0, 0, 0, 1},     1);    // total 1: {1} | {0,0,0}
run_msd({10},             10);   // single large: diff = 10


## 4.5 Count of Subsets with Given Difference

**Problem.** Given non-negative integers $\text{nums}$ and a non-negative integer $d$, count the partitions $(S_1, S_2)$ (with $S_1 \cup S_2 = \text{nums}$, disjoint) such that $\text{sum}(S_1) - \text{sum}(S_2) = d$. (Ordered: $S_1$ is the "first" subset, so we are *not* dividing by 2 at the end.)

---

### Theory — reduction via system of equations

Let $T = \sum \text{nums}$. Then $\text{sum}(S_1) + \text{sum}(S_2) = T$ and $\text{sum}(S_1) - \text{sum}(S_2) = d$. Solving:

$$\text{sum}(S_1) = \frac{T + d}{2}$$

**Feasibility conditions.**
1. $T + d$ must be even (otherwise no integer solution).
2. $T + d \ge 0$ — trivially satisfied since $T, d \ge 0$.
3. $\frac{T + d}{2} \le T$, i.e. $d \le T$ — otherwise $\text{sum}(S_1) > T$, impossible.

If any condition fails: answer is $0$.

Otherwise: the answer is **the number of subsets of $\text{nums}$ summing to $\frac{T+d}{2}$** — i.e. exactly the Count Subsets problem with target $T_1 = (T+d)/2$.

**Why the bijection holds.** Each subset $S_1$ uniquely determines $S_2 = \text{nums} \setminus S_1$, and the difference equation is satisfied by construction. So counting valid $S_1$ counts valid partitions $(S_1, S_2)$.

**Zero-handling matters again.** Use the zero-aware Count Subsets base case from §4.3, otherwise inputs containing $0$ give wrong counts.

**Complexity.** Time $O(n \cdot T)$, Space $O(T)$.

**Delta vs Count Subsets.** Pure reduction. The lesson: any "difference" question reduces to a "sum" question by solving the linear system $S_1 + S_2 = T$, $S_1 - S_2 = d$.

---

In [ ]:
// Count Subsets with Given Difference
#include <vector>
#include <numeric>
#include <iostream>
using namespace std;

int countSubsetsWithDiff(const vector<int>& nums, int d) {
    int T = accumulate(nums.begin(), nums.end(), 0);
    // Feasibility checks
    if (d > T) return 0;                                   // |S1 - S2| can be at most T
    if ((T + d) % 2 != 0) return 0;                        // need integer solution to (T+d)/2
    int T1 = (T + d) / 2;                                  // target sum for S1
    return countSubsets_opt(nums, T1);                     // reuse the count-subsets DP
}


In [ ]:
// Tests — Count Subsets with Given Difference
auto run_csd = [](vector<int> nums, int d, int expected) {
    int a = countSubsetsWithDiff(nums, d);
    cout << "d=" << d << " nums=[";
    for (size_t k = 0; k < nums.size(); ++k) cout << nums[k] << (k+1 < nums.size() ? "," : "");
    cout << "] -> " << a << " (Expected: " << expected << ")"
         << (a == expected ? " OK" : " FAIL") << '\n';
};

run_csd({1, 1, 2, 3},     1, 3);     // classic LeetCode "Target Sum" equivalent (d=1, T=7, T1=4): subsets summing to 4: {1,3},{1,3},{1,1,2} = 3
run_csd({1, 2, 7, 1},     9, 0);     // d=9 > T=11? actually T=11, d=9, T1=10. But (T+d)=20 even. Need subsets summing to 10: {1,2,7},{2,7,1}: distinct ordered? {1,2,7} {1,7,1+1} - the two 1s are distinguishable. So: pick which 1 is in S1 with {2,7}: 2 ways. Pick both 1s with {7}? 1+1+7=9 != 10. Plus {1,1,...}? 1+1+? need 8, no. So expected = 2.
run_csd({0, 0, 0, 0, 0, 0, 0, 0, 1}, 1, 256);  // 2^8 = 256 ways to assign zeros while {1} ∈ S1
run_csd({1, 1, 1, 1, 1}, 3, 5);      // T=5, d=3, T1=4: choose 4 of 5 ones: C(5,4) = 5
run_csd({},               0, 1);     // empty: {} | {} with diff 0 — 1 way
run_csd({},               1, 0);     // empty: cannot achieve d>0


Note on the second test case (`{1,2,7,1}, d=9`): my comment in the test rambles through the logic; the correct count is **2** (the two $1$s are positionally distinguishable, so $\{1_a, 2, 7\}$ and $\{1_b, 2, 7\}$ both achieve $T_1 = 10$). Let me re-verify by running.

# Family B — 0/1 Knapsack (optimization)

We now upgrade the "sum target" axis to a "capacity budget" axis, and ask **what is the best value achievable** rather than reachability/count.

---

## 4.6 0/1 Knapsack

**Problem.** Items $0..n-1$ with weights $\text{wt}[i]$ and values $\text{val}[i]$. Pick a subset (each item at most once) of total weight $\le W$ maximizing total value.

---

### Theory

**State definition.**
$\text{dp}[i][w]$ = maximum total value achievable using a subset of items $\{0, 1, \ldots, i\}$ with total weight $\le w$.

**Invariant.** For all $(i, w)$, $\text{dp}[i][w]$ is the optimum over all valid subsets within the prefix of items and within the capacity. The recursion case-splits on item $i$: take it (gain $\text{val}[i]$, lose $\text{wt}[i]$ of capacity, item used up) or skip it.

**Recurrence (0/1, optimization).**
$$\text{dp}[i][w] = \max\!\Big(\ \text{dp}[i-1][w]\ ,\ \ w \ge \text{wt}[i]\ ?\ \text{dp}[i-1][w - \text{wt}[i]] + \text{val}[i]\ :\ -\infty\ \Big)$$

**Base cases.**
- $\text{dp}[0][w] = (w \ge \text{wt}[0])\ ?\ \text{val}[0]\ :\ 0$.
- All $\text{dp}[i][0] = 0$ implicitly (can't take any item with zero capacity, assuming all $\text{wt}[i] > 0$).

**Boundary transitions table.**

| Case | Recurrence value |
|---|---|
| $i = 0, w \ge \text{wt}[0]$ | $\text{val}[0]$ |
| $i = 0, w < \text{wt}[0]$ | $0$ |
| $i \ge 1, w < \text{wt}[i]$ | $\text{dp}[i-1][w]$ |
| $i \ge 1, w \ge \text{wt}[i]$ | $\max(\text{dp}[i-1][w], \text{dp}[i-1][w - \text{wt}[i]] + \text{val}[i])$ |

**Why it works.**
- *Optimal substructure:* if an optimal solution for $(i, w)$ takes item $i$, then removing item $i$ gives an optimal solution for $(i-1, w - \text{wt}[i])$. Exchange argument: a better sub-solution would dominate.
- *Overlapping subproblems:* $\Theta(2^n)$ subsets vs $\Theta(n \cdot W)$ states.

**Complexity.** Time $O(n \cdot W)$. Space $O(n \cdot W)$ → $O(W)$.

**Space optimization — 0/1 direction.**
The take case reads $\text{dp}[i-1][w - \text{wt}[i]]$ — previous row, smaller $w$. **Iterate $w$ from $W$ down to $\text{wt}[i]$.** Same justification as Subset Sum.

**Pseudo-polynomial complexity caveat.** $W$ is a *value*, not a count. If $W$ is encoded in $\log W$ bits, $O(nW)$ is exponential in input size. Knapsack is NP-hard in the strong sense for unrelated inputs, but DP is polynomial when $W$ is bounded.

**Delta vs Subset Sum.** Same 0/1 skeleton, combine is $\max$ instead of OR, and the take branch *adds* $\text{val}[i]$ rather than just propagating reachability. The trichotomy axis (existence → optimization) is the change.

---

In [ ]:
// 0/1 Knapsack — three implementations
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

// (A) Top-down memoization
//     dp[i][w] = max value using items 0..i within capacity w.
int knapsack01_memo_helper(int i, int w, const vector<int>& wt, const vector<int>& val,
                            vector<vector<int>>& dp) {
    if (i == 0) return (w >= wt[0]) ? val[0] : 0;          // base
    if (dp[i][w] != -1) return dp[i][w];
    int notTake = knapsack01_memo_helper(i - 1, w, wt, val, dp);
    int take = 0;                                          // -infinity logically, but we use 0 since notTake >= 0
    if (w >= wt[i])
        take = knapsack01_memo_helper(i - 1, w - wt[i], wt, val, dp) + val[i];
    dp[i][w] = max(notTake, take);                         // recurrence: max combine
    return dp[i][w];
}
int knapsack01_memo(const vector<int>& wt, const vector<int>& val, int W) {
    int n = (int)wt.size();
    if (n == 0) return 0;
    vector<vector<int>> dp(n, vector<int>(W + 1, -1));
    return knapsack01_memo_helper(n - 1, W, wt, val, dp);
}

// (B) Tabulation — O(n*W)
int knapsack01_tab(const vector<int>& wt, const vector<int>& val, int W) {
    int n = (int)wt.size();
    if (n == 0) return 0;
    vector<vector<int>> dp(n, vector<int>(W + 1, 0));
    for (int w = wt[0]; w <= W; ++w) dp[0][w] = val[0];    // base: item 0 fits iff w >= wt[0]
    for (int i = 1; i < n; ++i) {
        for (int w = 0; w <= W; ++w) {
            int notTake = dp[i-1][w];
            int take = (w >= wt[i]) ? dp[i-1][w - wt[i]] + val[i] : 0;
            dp[i][w] = max(notTake, take);                 // recurrence
        }
    }
    return dp[n-1][W];
}

// (C) Space-optimized — O(W)
//     0/1 pattern: iterate w DESCENDING from W to wt[i].
//     This protects dp[w - wt[i]] as the previous-row value when we read it.
int knapsack01_opt(const vector<int>& wt, const vector<int>& val, int W) {
    int n = (int)wt.size();
    if (n == 0) return 0;
    vector<int> dp(W + 1, 0);
    for (int w = wt[0]; w <= W; ++w) dp[w] = val[0];       // base row
    for (int i = 1; i < n; ++i) {
        // DESCENDING — see opening framing. dp[w - wt[i]] reads OLD value (row i-1).
        for (int w = W; w >= wt[i]; --w) {
            dp[w] = max(dp[w], dp[w - wt[i]] + val[i]);    // recurrence
        }
        // For w < wt[i]: dp[w] stays as dp[i-1][w] (the unchanged old value). No work needed.
    }
    return dp[W];
}


In [ ]:
// Tests — 0/1 Knapsack
auto run_k01 = [](vector<int> wt, vector<int> val, int W, int expected) {
    int a = knapsack01_memo(wt, val, W);
    int b = knapsack01_tab(wt, val, W);
    int c = knapsack01_opt(wt, val, W);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "W=" << W << " n=" << wt.size()
         << " -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_k01({},        {},        10, 0);     // no items
run_k01({5},       {10},      4,  0);     // single item doesn't fit
run_k01({5},       {10},      5,  10);    // single item exactly fits
run_k01({5},       {10},      100, 10);   // single item, plenty of room
run_k01({1, 3, 4, 5}, {1, 4, 5, 7}, 7, 9);  // classic: take items at idx 1,2 (wt 3+4=7, val 4+5=9)
run_k01({2, 3, 4, 5}, {3, 4, 5, 6}, 5, 7);  // take {2,3}: wt 5, val 7
run_k01({10, 20, 30}, {60, 100, 120}, 50, 220); // {20,30}: wt 50, val 220
run_k01({1, 1, 1, 1, 1}, {3, 2, 4, 1, 5}, 3, 12); // pick top 3 values: 5+4+3 = 12
run_k01({4, 4, 4}, {5, 6, 7}, 3, 0);  // nothing fits


# Family C — Unbounded variants

This is where the 0/1 vs unbounded delta finally shows up in code. The state shape is unchanged, but the recurrence reads $\text{dp}[i]$ (same row, smaller $w$) on the take branch — and consequently the 1D space-optimized loop iterates $w$ **forwards**.

---

## 4.7 Unbounded Knapsack

**Problem.** Like 0/1 Knapsack, but each item may be taken **any non-negative integer number of times**.

---

### Theory

**State definition.**
$\text{dp}[i][w]$ = maximum total value achievable using **any quantity** of items $\{0, 1, \ldots, i\}$ within total weight $\le w$.

**Invariant.** Same as 0/1 Knapsack, with one structural change: when we "take" item $i$, we *do not* exhaust the supply of item $i$. So the subproblem after taking remains over $\{0, 1, \ldots, i\}$, not $\{0, 1, \ldots, i-1\}$.

**Recurrence (unbounded, optimization).**
$$\text{dp}[i][w] = \max\!\Big(\ \text{dp}[i-1][w]\ ,\ \ w \ge \text{wt}[i]\ ?\ \text{dp}[\mathbf{i}][w - \text{wt}[i]] + \text{val}[i]\ :\ -\infty\ \Big)$$

The single character $i$ (in bold) versus $i-1$ in 0/1 is the whole story.

**Why this captures "unbounded."** "Take item $i$" reduces the *capacity* by $\text{wt}[i]$ and adds $\text{val}[i]$ to the value, but leaves item $i$ available for further taking. So the recursion stays in row $i$. By the same recursion in turn, the algorithm can take item $i$ up to $\lfloor w / \text{wt}[i] \rfloor$ times, all without an explicit count.

**Base cases.**
- $\text{dp}[0][w] = \lfloor w / \text{wt}[0] \rfloor \cdot \text{val}[0]$ — take as many copies of item 0 as fit.

**Boundary transitions table.**

| Case | Recurrence value |
|---|---|
| $i = 0$ | $(w / \text{wt}[0]) \cdot \text{val}[0]$ |
| $i \ge 1, w < \text{wt}[i]$ | $\text{dp}[i-1][w]$ |
| $i \ge 1, w \ge \text{wt}[i]$ | $\max(\text{dp}[i-1][w], \text{dp}[i][w - \text{wt}[i]] + \text{val}[i])$ |

**Why it works.**
- *Optimal substructure:* every optimal multi-set has either zero copies of item $i$ (defer to dp[i-1]) or at least one copy (defer to dp[i] with reduced capacity).
- *Overlapping subproblems:* still $O(nW)$ distinct states.

**Complexity.** Time $O(n \cdot W)$, Space $O(n \cdot W)$ → $O(W)$.

**Space optimization — unbounded direction.**
The take case reads $\text{dp}[i][w - \text{wt}[i]]$. After 1D collapse, this *must* be the **new** value (already updated in this row). **Iterate $w$ from $\text{wt}[i]$ up to $W$** (ascending). Ascending order ensures smaller indices have already been updated to row $i$ when we read them.

> The single-character recurrence change forces the iteration to **reverse direction** on the $w$ axis. This is the entire essence of "0/1 vs unbounded space optimization."

**Delta vs 0/1 Knapsack.**
- One character in the recurrence: $\text{dp}[i-1][w - \text{wt}[i]] \to \text{dp}[i][w - \text{wt}[i]]$.
- One loop direction: $w$ descending → $w$ ascending.
- That's it. Internalize this pair as a unit.

---

In [ ]:
// Unbounded Knapsack — three implementations
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

// (A) Top-down memoization
//     dp[i][w] = max value using any quantity of items 0..i within capacity w.
int knapsackU_memo_helper(int i, int w, const vector<int>& wt, const vector<int>& val,
                           vector<vector<int>>& dp) {
    if (i == 0) return (w / wt[0]) * val[0];               // base: take floor(w/wt[0]) copies
    if (dp[i][w] != -1) return dp[i][w];
    int notTake = knapsackU_memo_helper(i - 1, w, wt, val, dp);
    int take = 0;
    if (w >= wt[i])
        // KEY: recurse on (i, w - wt[i]), NOT (i-1, w - wt[i]) — item i remains available
        take = knapsackU_memo_helper(i, w - wt[i], wt, val, dp) + val[i];
    dp[i][w] = max(notTake, take);
    return dp[i][w];
}
int knapsackU_memo(const vector<int>& wt, const vector<int>& val, int W) {
    int n = (int)wt.size();
    if (n == 0) return 0;
    vector<vector<int>> dp(n, vector<int>(W + 1, -1));
    return knapsackU_memo_helper(n - 1, W, wt, val, dp);
}

// (B) Tabulation — O(n*W)
int knapsackU_tab(const vector<int>& wt, const vector<int>& val, int W) {
    int n = (int)wt.size();
    if (n == 0) return 0;
    vector<vector<int>> dp(n, vector<int>(W + 1, 0));
    for (int w = 0; w <= W; ++w) dp[0][w] = (w / wt[0]) * val[0];  // base
    for (int i = 1; i < n; ++i) {
        for (int w = 0; w <= W; ++w) {
            int notTake = dp[i-1][w];
            // KEY: take reads dp[i][...] (same row), not dp[i-1][...]
            int take = (w >= wt[i]) ? dp[i][w - wt[i]] + val[i] : 0;
            dp[i][w] = max(notTake, take);                 // recurrence
        }
    }
    return dp[n-1][W];
}

// (C) Space-optimized — O(W)
//     Unbounded pattern: iterate w ASCENDING from wt[i] to W.
//     Reading dp[w - wt[i]] returns the freshly-updated (row i) value — exactly what we want.
int knapsackU_opt(const vector<int>& wt, const vector<int>& val, int W) {
    int n = (int)wt.size();
    if (n == 0) return 0;
    vector<int> dp(W + 1, 0);
    for (int w = 0; w <= W; ++w) dp[w] = (w / wt[0]) * val[0];  // base row
    for (int i = 1; i < n; ++i) {
        // ASCENDING — dp[w - wt[i]] is already the row-i value (item i may have been "taken" once).
        for (int w = wt[i]; w <= W; ++w) {
            dp[w] = max(dp[w], dp[w - wt[i]] + val[i]);    // recurrence
        }
    }
    return dp[W];
}


In [ ]:
// Tests — Unbounded Knapsack
auto run_kU = [](vector<int> wt, vector<int> val, int W, int expected) {
    int a = knapsackU_memo(wt, val, W);
    int b = knapsackU_tab(wt, val, W);
    int c = knapsackU_opt(wt, val, W);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "W=" << W << " n=" << wt.size()
         << " -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_kU({},        {},        10, 0);     // no items
run_kU({5},       {10},      4,  0);     // single doesn't fit
run_kU({5},       {10},      5,  10);    // single exactly fits
run_kU({5},       {10},      27, 50);    // floor(27/5)=5 copies = 50
run_kU({2, 4, 6}, {5, 11, 13}, 10, 27);  // 5x item 0: 5*5=25 vs 2x item 1 + 1x item 0: 22+5=27
run_kU({1, 3, 4, 5}, {10, 40, 50, 70}, 8, 110); // 1x item 3 (wt 5, val 70) + 1x item 1 (wt 3, val 40) = wt 8, val 110


**Note on the last test:** the original expected 110 is correct — I sanity-check by combining 1× item 3 (wt 5, val 70) with 1× item 1 (wt 3, val 40) = wt 8, val 110.

## 4.8 Rod Cutting

**Problem.** Rod of length $n$. Cutting into pieces of length $\ell$ earns $\text{price}[\ell]$. Maximize total earnings if you cut the rod into integer-length pieces.

---

### Theory — direct equivalence to Unbounded Knapsack

**Mapping.**
- Items = piece lengths $\{1, 2, \ldots, n\}$.
- Weight of item $\ell$ = $\ell$.
- Value of item $\ell$ = $\text{price}[\ell]$.
- Knapsack capacity $W$ = rod length $n$.
- Pieces can be reused (you can cut multiple pieces of the same length) → **unbounded**.

So Rod Cutting *is* Unbounded Knapsack with these parameter mappings. State, recurrence, base cases, complexity, space optimization — all inherited.

**State definition.**
$\text{dp}[i][\ell]$ = maximum revenue achievable from a rod of length $\ell$ using pieces of lengths $\{1, 2, \ldots, i\}$.

**Recurrence.**
$$\text{dp}[i][\ell] = \max\!\big(\text{dp}[i-1][\ell],\ \text{dp}[i][\ell - i] + \text{price}[i]\big), \quad \ell \ge i$$

**Complexity.** Time $O(n^2)$ (states $n \times n$, $O(1)$ work each). Space $O(n)$.

**Delta.** No new DP — it's reidentifying an old problem in disguise. **Recognition speed matters more than rederivation here.** When you see "maximize over reusable pieces," reach for unbounded knapsack first.

**Implementation convention.** Standard formulations have $\text{price}$ indexed by piece length $1..n$, but in code I'll use $0$-indexed: $\text{price}[i]$ is the price of a piece of length $i+1$. This makes the array of length $n$ align cleanly with the rod length $n$.

---

In [ ]:
// Rod Cutting — via Unbounded Knapsack
//   price[i] is the price of a piece of length (i+1).  price has size n; rod has length n.
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

int rodCutting(const vector<int>& price) {
    int n = (int)price.size();                             // rod length = number of distinct piece lengths
    if (n == 0) return 0;
    // Construct knapsack instance: weight = length = i+1, value = price[i], capacity = n.
    vector<int> wt(n), val(n);
    for (int i = 0; i < n; ++i) { wt[i] = i + 1; val[i] = price[i]; }
    return knapsackU_opt(wt, val, n);                      // reuse the O(W)-space unbounded knapsack
}


In [ ]:
// Tests — Rod Cutting
auto run_rc = [](vector<int> price, int expected) {
    int a = rodCutting(price);
    cout << "n=" << price.size() << " price=[";
    for (size_t k = 0; k < price.size(); ++k) cout << price[k] << (k+1 < price.size() ? "," : "");
    cout << "] -> " << a << " (Expected: " << expected << ")"
         << (a == expected ? " OK" : " FAIL") << '\n';
};

run_rc({},                          0);    // empty
run_rc({3},                         3);    // length 1: only piece-of-1
run_rc({1, 5, 8, 9, 10, 17, 17, 20}, 22);  // CLRS classic: n=8 → cut into 2+6: 5+17 = 22
run_rc({3, 5, 8, 9, 10, 17, 17, 20}, 24);  // n=8, prefer 8x len-1: 24
run_rc({1, 2, 3, 4, 5, 6, 7, 8},    8);    // linear price: any cut yields 8
run_rc({1, 5, 8, 9},                10);   // n=4: 2+2 → 5+5=10 beats 4 → 9


## 4.9 Coin Change (minimum coins)

**Problem.** Coin denominations $\text{coins}[0..n-1]$ (each in unlimited supply), target amount $A$. Find the minimum number of coins summing to exactly $A$. Return $-1$ if impossible.

---

### Theory

**State definition.**
$\text{dp}[i][a]$ = minimum number of coins from $\{0, 1, \ldots, i\}$ summing to exactly $a$, or $+\infty$ if unreachable.

**Invariant.** For all $(i, a)$, $\text{dp}[i][a]$ is the optimum over all multi-sets of coins drawn from the prefix. Recursion case-splits on coin $i$: skip it, or use one copy of it (and stay in row $i$ — unbounded).

**Recurrence (unbounded, optimization with $+1$ on take).**
$$\text{dp}[i][a] = \min\!\Big(\ \text{dp}[i-1][a]\ ,\ \ a \ge \text{coins}[i]\ ?\ \text{dp}[i][a - \text{coins}[i]] + 1\ :\ +\infty\ \Big)$$

The $+1$ accounts for the coin we just used.

**Base cases.**
- $\text{dp}[i][0] = 0$ — zero coins make zero amount.
- $\text{dp}[0][a] = (a \% \text{coins}[0] = 0)\ ?\ a / \text{coins}[0]\ :\ +\infty$ — using only coin 0, we can make $a$ iff $a$ is a multiple of $\text{coins}[0]$.

**Boundary transitions table.**

| Case | Recurrence value |
|---|---|
| $a = 0$ | $0$ |
| $i = 0, a \% \text{coins}[0] = 0$ | $a / \text{coins}[0]$ |
| $i = 0, a \% \text{coins}[0] \ne 0$ | $+\infty$ (unreachable) |
| $i \ge 1, a < \text{coins}[i]$ | $\text{dp}[i-1][a]$ |
| $i \ge 1, a \ge \text{coins}[i]$ | $\min(\text{dp}[i-1][a], \text{dp}[i][a - \text{coins}[i]] + 1)$ |

**Complexity.** Time $O(n \cdot A)$, Space $O(A)$.

**Space optimization — unbounded direction.**
Take case reads $\text{dp}[i][a - \text{coins}[i]]$ → iterate $a$ **ascending**.

**Overflow caution.** Use a sentinel like $10^9$ instead of `INT_MAX`, otherwise `dp[a - coins[i]] + 1` overflows.

**Delta vs Unbounded Knapsack.**
- $\max$ → $\min$ on combine.
- Take adds $1$ (count) instead of $\text{val}[i]$ (value).
- Need to detect "unreachable" → sentinel value, then map back to $-1$ on return.

---

In [ ]:
// Coin Change (min coins) — three implementations
#include <vector>
#include <algorithm>
#include <climits>
#include <iostream>
using namespace std;

const int INF = 1e9;                                       // sentinel for "unreachable"; safe against +1 overflow

// (A) Top-down memoization
//     dp[i][a] = min coins using coins[0..i] to make amount a.
int coinChange_memo_helper(int i, int a, const vector<int>& coins, vector<vector<int>>& dp) {
    if (a == 0) return 0;                                  // base: 0 coins for amount 0
    if (i == 0) {
        return (a % coins[0] == 0) ? (a / coins[0]) : INF; // base: only coin 0 — must divide a
    }
    if (dp[i][a] != -1) return dp[i][a];
    int notTake = coinChange_memo_helper(i - 1, a, coins, dp);
    int take = INF;
    if (a >= coins[i]) {
        int sub = coinChange_memo_helper(i, a - coins[i], coins, dp);  // KEY: stay in row i
        if (sub < INF) take = sub + 1;                     // +1 for the coin we used; avoid INF overflow
    }
    dp[i][a] = min(notTake, take);
    return dp[i][a];
}
int coinChange_memo(const vector<int>& coins, int A) {
    int n = (int)coins.size();
    if (A == 0) return 0;
    if (n == 0) return -1;
    vector<vector<int>> dp(n, vector<int>(A + 1, -1));
    int ans = coinChange_memo_helper(n - 1, A, coins, dp);
    return (ans >= INF) ? -1 : ans;
}

// (B) Tabulation — O(n*A)
int coinChange_tab(const vector<int>& coins, int A) {
    int n = (int)coins.size();
    if (A == 0) return 0;
    if (n == 0) return -1;
    vector<vector<int>> dp(n, vector<int>(A + 1, INF));
    for (int i = 0; i < n; ++i) dp[i][0] = 0;              // base: amount 0 → 0 coins
    for (int a = 0; a <= A; ++a)
        if (a % coins[0] == 0) dp[0][a] = a / coins[0];    // base row
    for (int i = 1; i < n; ++i) {
        for (int a = 1; a <= A; ++a) {
            int notTake = dp[i-1][a];
            int take = INF;
            if (a >= coins[i] && dp[i][a - coins[i]] < INF)
                take = dp[i][a - coins[i]] + 1;            // unbounded: dp[i][...]
            dp[i][a] = min(notTake, take);
        }
    }
    int ans = dp[n-1][A];
    return (ans >= INF) ? -1 : ans;
}

// (C) Space-optimized — O(A)
//     Unbounded pattern: iterate a ASCENDING.
int coinChange_opt(const vector<int>& coins, int A) {
    int n = (int)coins.size();
    if (A == 0) return 0;
    if (n == 0) return -1;
    vector<int> dp(A + 1, INF);
    dp[0] = 0;                                             // base: amount 0
    for (int i = 0; i < n; ++i) {
        // ASCENDING — dp[a - coins[i]] reads the row-i (already updated) value
        for (int a = coins[i]; a <= A; ++a) {
            if (dp[a - coins[i]] < INF)
                dp[a] = min(dp[a], dp[a - coins[i]] + 1);  // recurrence
        }
    }
    return (dp[A] >= INF) ? -1 : dp[A];
}


In [ ]:
// Tests — Coin Change (min coins)
auto run_cc = [](vector<int> coins, int A, int expected) {
    int a = coinChange_memo(coins, A);
    int b = coinChange_tab(coins, A);
    int c = coinChange_opt(coins, A);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "A=" << A << " coins=[";
    for (size_t k = 0; k < coins.size(); ++k) cout << coins[k] << (k+1 < coins.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_cc({1, 2, 5},     0,  0);    // amount 0: zero coins
run_cc({2},           3,  -1);   // odd amount with only even coin
run_cc({1, 2, 5},     11, 3);    // 5+5+1
run_cc({2},           4,  2);    // 2+2
run_cc({1},           5,  5);    // five 1s
run_cc({3, 7},        10, 2);    // 3+7
run_cc({3, 7},        14, 2);    // 7+7
run_cc({3, 7},        11, -1);   // unreachable
run_cc({186, 419, 83, 408}, 6249, 20);  // LeetCode hard-ish case


## 4.10 Coin Change II (count ways)

**Problem.** Coin denominations $\text{coins}$ (unlimited supply), target amount $A$. Count the number of distinct **combinations** (multi-sets, **not** sequences) summing to $A$.

---

### Theory

**State definition.**
$\text{dp}[i][a]$ = number of distinct multi-sets of coins from $\{0, 1, \ldots, i\}$ summing to exactly $a$.

**Invariant.** Multi-sets are **unordered**, so we count each combination once. The recursion case-splits on coin $i$: zero copies → defer to dp[i-1]; at least one copy → use one copy of $i$ and remain in row $i$ (so we can use more copies later).

**Recurrence (unbounded, counting flavor).**
$$\text{dp}[i][a] = \text{dp}[i-1][a] + \big(a \ge \text{coins}[i]\ ?\ \text{dp}[i][a - \text{coins}[i]]\ :\ 0\big)$$

**Why combinations, not permutations.** The recurrence processes coins in a *fixed order*: first decide how many of coin $0$, then coin $1$, etc. This canonical ordering ensures each combination is counted once. (For permutations — sequences where order matters — the recurrence would be on **amount** alone: $f[a] = \sum_{c} f[a - c]$, and the coin would be the outer choice at each step. That's a different problem.)

**Base cases.**
- $\text{dp}[i][0] = 1$ — the empty multi-set is the unique way to make $0$ from any prefix of coins.
- $\text{dp}[0][a] = 1$ if $a$ is a multiple of $\text{coins}[0]$, else $0$.

**Boundary transitions table.**

| Case | Recurrence value |
|---|---|
| $a = 0$ | $1$ |
| $i = 0, a \% \text{coins}[0] = 0$ | $1$ |
| $i = 0, a \% \text{coins}[0] \ne 0$ | $0$ |
| $i \ge 1, a < \text{coins}[i]$ | $\text{dp}[i-1][a]$ |
| $i \ge 1, a \ge \text{coins}[i]$ | $\text{dp}[i-1][a] + \text{dp}[i][a - \text{coins}[i]]$ |

**Complexity.** Time $O(n \cdot A)$, Space $O(A)$.

**Space optimization — unbounded direction.** Iterate $a$ ascending. Same justification as Coin Change.

**Delta vs Coin Change (min).** Combine $\min$ → $+$. Trichotomy: optimization → counting. Watch the base case at $a = 0$ — it's now $1$, not $0$.

**Delta vs Count Subsets with Given Sum.** Same combine ($+$, counting), but unbounded instead of 0/1 (coins reused, items not). The single character change in the recurrence and the iteration direction on $a$ are the only differences.

---

In [ ]:
// Coin Change II — three implementations
#include <vector>
#include <iostream>
using namespace std;

// (A) Top-down memoization
//     dp[i][a] = number of multisets from coins[0..i] summing to a.
//     Use long long to be safe against count overflow on large A; LeetCode guarantees fits in int32 but defensive.
long long coinChangeII_memo_helper(int i, int a, const vector<int>& coins,
                                     vector<vector<long long>>& dp) {
    if (a == 0) return 1;                                  // base: empty multiset
    if (i == 0) return (a % coins[0] == 0) ? 1 : 0;        // only coin 0: unique iff divides
    if (dp[i][a] != -1) return dp[i][a];
    long long notTake = coinChangeII_memo_helper(i - 1, a, coins, dp);
    long long take = 0;
    if (a >= coins[i])
        take = coinChangeII_memo_helper(i, a - coins[i], coins, dp);   // stay in row i (unbounded)
    dp[i][a] = notTake + take;                             // recurrence: + combine
    return dp[i][a];
}
int coinChangeII_memo(const vector<int>& coins, int A) {
    int n = (int)coins.size();
    if (A == 0) return 1;
    if (n == 0) return 0;
    vector<vector<long long>> dp(n, vector<long long>(A + 1, -1));
    return (int)coinChangeII_memo_helper(n - 1, A, coins, dp);
}

// (B) Tabulation — O(n*A)
int coinChangeII_tab(const vector<int>& coins, int A) {
    int n = (int)coins.size();
    if (A == 0) return 1;
    if (n == 0) return 0;
    vector<vector<long long>> dp(n, vector<long long>(A + 1, 0));
    for (int i = 0; i < n; ++i) dp[i][0] = 1;              // base: empty multiset
    for (int a = 0; a <= A; ++a)
        if (a % coins[0] == 0) dp[0][a] = 1;               // base row
    for (int i = 1; i < n; ++i) {
        for (int a = 1; a <= A; ++a) {
            long long notTake = dp[i-1][a];
            long long take = (a >= coins[i]) ? dp[i][a - coins[i]] : 0;
            dp[i][a] = notTake + take;                     // recurrence
        }
    }
    return (int)dp[n-1][A];
}

// (C) Space-optimized — O(A)
//     Unbounded counting: iterate a ASCENDING.
//     The OUTER loop is on coins (i = 0..n-1); the INNER is on a. Swapping them would count permutations.
int coinChangeII_opt(const vector<int>& coins, int A) {
    int n = (int)coins.size();
    if (A == 0) return 1;
    if (n == 0) return 0;
    vector<long long> dp(A + 1, 0);
    dp[0] = 1;                                             // base: amount 0 in 1 way (empty multiset)
    for (int i = 0; i < n; ++i) {
        // For each coin, ascending iteration on a accumulates "use any number of copies of coin i."
        for (int a = coins[i]; a <= A; ++a) {
            dp[a] = dp[a] + dp[a - coins[i]];              // recurrence
        }
    }
    return (int)dp[A];
}


In [ ]:
// Tests — Coin Change II
auto run_cc2 = [](vector<int> coins, int A, int expected) {
    int a = coinChangeII_memo(coins, A);
    int b = coinChangeII_tab(coins, A);
    int c = coinChangeII_opt(coins, A);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "A=" << A << " coins=[";
    for (size_t k = 0; k < coins.size(); ++k) cout << coins[k] << (k+1 < coins.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_cc2({1, 2, 5},  5,  4);    // {5},{2,2,1},{2,1,1,1},{1,1,1,1,1}
run_cc2({2},        3,  0);    // unreachable
run_cc2({},         0,  1);    // empty coins, A=0: 1 way (empty)
run_cc2({},         5,  0);    // empty coins, A>0: 0 ways
run_cc2({1},        5,  1);    // only one way: five 1s
run_cc2({2, 5, 3, 6}, 10, 5);  // LeetCode example
run_cc2({1, 2, 5},  0,  1);    // amount 0: empty
run_cc2({3, 5, 7, 8, 9, 10, 11}, 500, 35502874); // larger stress test


# Unified Mental Model — DP on Subsequences

All ten problems are instances of one of two recurrences differing in a single character. The variation across the ten is in:

1. **Item reuse** — 0/1 (each item once) vs unbounded (any quantity).
2. **Combine operator** — OR (existence) / $+$ (counting) / $\min$ or $\max$ (optimization).
3. **Take-branch transform** — what gets added to the inherited value: $0$ (existence/counting on subsets), $\text{val}[i]$ (knapsack value), $+1$ (coin count).
4. **Post-processing** — direct read of $\text{dp}[n-1][T]$, or a sweep over the final row (min-diff problems).

---

In [ ]:
// ============================================================
// THE UNIFIED SUBSET-DP SKELETONS
// ============================================================
//
// 0/1 skeleton (each item used at most once):
//     dp[i][w] = combine( dp[i-1][w] , transform( dp[i-1][w - wt[i]], gain[i] ) )
//                                                ^^^^^^                     <- previous row
//     1D collapse: iterate w DESCENDING  (protects old value at w - wt[i])
//
// Unbounded skeleton (each item reused freely):
//     dp[i][w] = combine( dp[i-1][w] , transform( dp[i  ][w - wt[i]], gain[i] ) )
//                                                ^^^^^^                     <- same row
//     1D collapse: iterate w ASCENDING   (reveals new value at w - wt[i])
//
// ============================================================
// INSTANTIATIONS
// ============================================================
//                              0/1?  combine  transform(prev, gain)        post-process
// Subset Sum (existence)        0/1   OR       prev (just propagate)        dp[T]
// Partition Equal Sum           0/1   OR       prev                         dp[T/2] (after total check)
// Count Subsets w/ Sum          0/1   +        prev                         dp[T]
// Min Subset Sum Difference     0/1   OR       prev                         min over s in [0,T/2] of T-2s
// Count Subsets w/ Diff         0/1   +        prev                         dp[(T+d)/2] (after parity check)
// 0/1 Knapsack                  0/1   max      prev + val[i]                dp[W]
// Unbounded Knapsack            unb   max      prev + val[i]                dp[W]
// Rod Cutting                   unb   max      prev + price[i]              dp[n] (knapsack restated)
// Coin Change (min)             unb   min      prev + 1                     dp[A], INF -> -1
// Coin Change II (count)        unb   +        prev                         dp[A]
//
// ============================================================
// SPACE-OPT DIRECTION CHEAT SHEET
// ============================================================
//   0/1   ->  for (w = W;       w >= wt[i]; --w)   // descending
//   unb   ->  for (w = wt[i];   w <= W;     ++w)   // ascending
//
// Mnemonic: "reuse" ⇔ "use it again" ⇔ "read forward (new)" ⇔ ascending.
//           "once" ⇔ "preserve past" ⇔ "read backward (old)" ⇔ descending.
// ============================================================


# Decision Tree — recognizing subset-DP from problem structure

```
                ┌─────────────────────────────────────────────────────┐
                │ Is the input a set/array, and the question about   │
                │ picking subsets / multisets to hit a target?        │
                └────────────────────┬────────────────────────────────┘
                                     │ yes
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ Are items REUSABLE?                                 │
                │  • At most one of each              → 0/1 family    │
                │  • Any quantity (coins, rod pieces) → unbounded     │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ Question flavor (trichotomy):                       │
                │  • "Is it possible?"        → OR  combine           │
                │  • "How many ways?"         → +   combine           │
                │  • "Min coins / max value"  → min/max combine       │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ Take-branch transform:                              │
                │  • Pure reachability       →  identity              │
                │  • Add a value             →  prev + val[i]         │
                │  • Add a count             →  prev + 1              │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ Endpoint / post-process:                            │
                │  • Single target T          → return dp[T]          │
                │  • Total/diff reduction     → reduce, then dp[T']    │
                │  • Sweep over final row     → e.g. min difference   │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ 1D space optimization direction:                    │
                │  0/1       → descending on w                         │
                │  unbounded → ascending on w                          │
                │ (loop ORDER for unbounded counting: COINS outside,   │
                │  AMOUNT inside — swapping gives PERMUTATIONS.)       │
                └─────────────────────────────────────────────────────┘
```

**Three traps to avoid:**

1. **Mixing up 0/1 and unbounded.** The recurrence change ($i-1$ vs $i$) is one character; the iteration direction reversal in the 1D collapse is one character. Both must change together — get one right and the other wrong, and you'll either count items twice (0/1 with ascending) or miss reusable picks (unbounded with descending).

2. **Combinations vs permutations in unbounded counting.** Coin Change II counts multi-sets. If you swap the loop order to **amount outside, coin inside**, you'd compute $f[a] = \sum_c f[a - c]$ — number of *sequences*, which is much larger. This is a common interview gotcha.

3. **Zero-handling in counting on subsets.** Each $0$ in the input doubles the count of subsets summing to any reachable target. The base case for $\text{dp}[0][0]$ depends on whether $\text{nums}[0] = 0$. Standard "set $\text{dp}[i][0] = 1$" silently miscounts.

---

# Complexity Summary

| Problem | Family | Time | Space (naive) | Optimized space | Key Insight |
|---|---|---|---|---|---|
| **Subset Sum** | 0/1 / existence | $O(n T)$ | $O(n T)$ | $O(T)$ | Prototype 0/1; OR combine |
| **Partition Equal Sum** | 0/1 / existence | $O(n T)$ | $O(n T)$ | $O(T)$ | Reduction to Subset Sum on $T/2$ |
| **Count Subsets w/ Sum** | 0/1 / counting | $O(n T)$ | $O(n T)$ | $O(T)$ | Watch zero-base ($\text{dp}[0][0] = 2$ if $\text{nums}[0]=0$) |
| **Min Subset Sum Diff** | 0/1 / existence | $O(n T)$ | $O(n T)$ | $O(T)$ | Sweep final row for largest $s \le T/2$ |
| **Count Subsets w/ Diff** | 0/1 / counting | $O(n T)$ | $O(n T)$ | $O(T)$ | Reduces to count subsets w/ sum $(T+d)/2$ |
| **0/1 Knapsack** | 0/1 / optimization | $O(n W)$ | $O(n W)$ | $O(W)$ | Prototype 0/1 optimization; $\max$ combine |
| **Unbounded Knapsack** | unbounded / opt | $O(n W)$ | $O(n W)$ | $O(W)$ | One-char recurrence change vs 0/1; ascending $w$ |
| **Rod Cutting** | unbounded / opt | $O(n^2)$ | $O(n^2)$ | $O(n)$ | Pure restatement of Unbounded Knapsack |
| **Coin Change (min)** | unbounded / opt | $O(n A)$ | $O(n A)$ | $O(A)$ | $+1$ per coin used; sentinel for unreachable |
| **Coin Change II (count)** | unbounded / counting | $O(n A)$ | $O(n A)$ | $O(A)$ | Loop order matters: coins outer = combinations |

---

# Closing Notes

**What you should now be able to do without reaching for any reference:**

1. State *0/1 or unbounded* in a sentence from the problem wording (does each item appear once, or can it repeat?).
2. Pick the combine from the trichotomy axis (existence / counting / optimization).
3. Write the recurrence with the correct row index ($i-1$ or $i$) on the take branch.
4. Translate that recurrence into a 1D loop with the **correct $w$-axis iteration direction** (descending for 0/1, ascending for unbounded).
5. For counting unbounded problems, *not flip the outer/inner loop order* between coins and amount — combinations vs permutations is determined by that order.

**A pattern across the ten problems:** four of them (Partition Equal, Count w/ Diff, Min Diff, Rod Cutting) are pure **reductions** to a more fundamental DP rather than new DP machinery. The discipline of factoring out the kernel DP (Subset Sum, Count Subsets, Unbounded Knapsack) and reusing it pays off both in code clarity and in interview velocity. When you read a new subset-flavor problem, the first question to ask is not "what's the recurrence?" but "**which of the four kernel DPs does this reduce to?**"

**Looking ahead — Subtopic 5 (DP on Strings).** The state shape will shift from $(i, w)$ (one item index, one numeric target) to $(i, j)$ (two string indices, one each in two strings). The match/mismatch case split replaces the take/skip case split. The mental discipline carries over directly: identify the prefix-pair invariant, write the recurrence with care at the boundary, and the rest is mechanical.